# Team A — Final evaluation: training score, 2026 test and error analysis

This notebook produces every number reported in the **Model Evaluation Metrics** and
**Solution Design** forms. It **does not train or tune** any model. It loads the frozen models
saved by `run_fixed_pipeline.ps1` and scores them.

**Data:** original Kaggle training data (2019–2020) plus 2021–2025 synthetic transactions for
999 new customers, regenerated with the corrected Sparkov sampler (`sparkov_fixes.py`). The
first generation had a sampling bug that tied purchase category to date, so each year contained
only 4–6 legitimate categories. The 2026 file is the reserved test set; its labels are first read
here, after all models and thresholds were saved.

| Model | Saved folder | Role |
|---|---|---|
| Gradient Boosting + 17 history features | `Models/expanded_fixed_run1/history_v3` | **Final model** |
| Gradient Boosting, transaction-only | `Models/expanded_fixed_run1/transaction_models` | Comparison |
| Random Forest | `Models/expanded_fixed_run1/rf_baseline` | Comparison baseline |
| Majority class | none | Simple baseline |

All thresholds were chosen on the chronological validation period using the rule fixed in
Part II (maximum F1 with recall ≥ 0.80 and precision ≥ 0.10).
The corrected generation followed investigation of the earlier defective test. Thus this regenerated chronological test is not independent of the overall development history. No model or threshold was changed after its corrected test result.


## 1. Setup

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT))
import project_paths
os.chdir(PROJECT)
from training_score import training_score
from evaluate import evaluate

DATA_DIR = PROJECT / 'Data/expanded_training_fixed'
ART = PROJECT / 'Models/expanded_fixed_run1'
MODELS = {
    'GB + history (final)': ART / 'history_v3',
    'GB transaction-only': ART / 'transaction_models',
    'Random Forest': ART / 'rf_baseline',
}
FINAL = MODELS['GB + history (final)']
for name, folder in MODELS.items():
    meta = json.loads((folder / 'experiment.json').read_text())
    print(f"{name}: {meta['selected_model']}, threshold {meta['threshold']:.4f}, {meta['feature_count']} features")
print('Data folder found:', DATA_DIR.exists())

GB + history (final): Gradient Boosting, threshold 0.9610, 95 features
GB transaction-only: Gradient Boosting, threshold 0.9623, 78 features
Random Forest: Random Forest, threshold 0.6709, 78 features
Data folder found: True


## 2. Training score (overfitting check)

Each saved model scores **its own fitting rows** (the earliest ~80% of development transactions)
at its saved threshold. The split is rebuilt exactly as in `train.py` and checked against the
locked experiment. Results are saved as `training_score.json`; if that file already exists
it is read instead of recomputed. The history model takes the longest (history features for 5 million rows).

In [2]:
train_scores = {}
for name, folder in MODELS.items():
    saved = folder / 'training_score.json'
    if saved.exists():
        train_scores[name] = json.loads(saved.read_text())
        print(f'{name}: read saved training score')
    else:
        print(f'--- {name} ---')
        train_scores[name] = training_score(folder, DATA_DIR)

GB + history (final): read saved training score
GB transaction-only: read saved training score
Random Forest: read saved training score


## 3. 2026 test evaluation

`evaluate.py` scores each frozen model once on the reserved 2026 transactions and writes
`final_test.json`, `test_confusion.png`, `test_subgroups.csv` and row-level `test_scores.csv`.
History scoring uses all earlier transactions (2019–2025, labels not used) as context.
Existing records are read, not rerun.

In [3]:
test_scores = {}
for name, folder in MODELS.items():
    saved = folder / 'final_test.json'
    if saved.exists():
        test_scores[name] = json.loads(saved.read_text())
        print(f'{name}: read saved 2026 evaluation')
    else:
        print(f'--- {name} ---')
        test_scores[name] = evaluate(folder, DATA_DIR)

GB + history (final): read saved 2026 evaluation
GB transaction-only: read saved 2026 evaluation
Random Forest: read saved 2026 evaluation


## 4. Validation comparison of all candidates (model selection)

Every candidate was scored on the same chronological validation rows. This table supports the
model-selection section of the Solution Design form.

In [4]:
validation = pd.concat([pd.read_csv(MODELS['Random Forest'] / 'validation_comparison.csv'),
                        pd.read_csv(MODELS['GB transaction-only'] / 'validation_comparison.csv'),
                        pd.read_csv(FINAL / 'validation_comparison.csv').assign(
                            model=lambda t: t.model.where(t.model != 'Gradient Boosting', 'Gradient Boosting + history'))])
validation = validation.drop_duplicates('model')[['model', 'threshold', 'precision', 'recall', 'f1',
                                                   'average_precision', 'fp', 'fn', 'fit_seconds', 'targets_met']]
validation.to_csv(PROJECT / 'Part_III' / 'Results' / 'fixed_validation_comparison.csv', index=False)
validation.round(4)

,model,threshold,precision,recall,f1,average_precision,fp,fn,fit_seconds,targets_met
0,Majority class,0.5000,0.0000,0.0000,0.0000,0.0019,0,2430,0.0000,False
1,Amount rule,514.1916,0.1143,0.4794,0.1846,0.0869,9025,1265,0.0000,False
2,Random Forest,0.6709,0.4305,0.8000,0.5597,0.6598,2572,486,616.3452,True
2,Logistic Regression,0.8136,0.1073,0.6757,0.1852,0.0745,13659,788,67.2660,False
3,Gradient Boosting,0.9623,0.4950,0.8000,0.6116,0.7537,1983,486,97.3392,True
2,Gradient Boosting + history,0.9610,0.9518,0.8770,0.9128,0.9580,108,299,118.2842,True


## 5. Training vs. validation vs. 2026 test

In [5]:
def validation_row(folder, model):
    table = pd.read_csv(folder / 'validation_comparison.csv')
    return table[table.model == model].iloc[0].to_dict()

rows = []
for name, folder in MODELS.items():
    model = json.loads((folder / 'experiment.json').read_text())['selected_model']
    for split, record in [('Training', train_scores[name]), ('Validation', validation_row(folder, model)),
                          ('2026 test', test_scores[name])]:
        rows.append({'model': name, 'split': split, 'rows': int(record['rows']), 'fraud': int(record['fraud']),
                     'precision': record['precision'], 'recall': record['recall'], 'f1': record['f1'],
                     'pr_auc': record['average_precision'], 'false_alerts': int(record['fp']),
                     'missed_fraud': int(record['fn'])})
# Majority-class baseline on 2026: flags nothing.
t = test_scores['GB + history (final)']
rows.append({'model': 'Majority class', 'split': '2026 test', 'rows': t['rows'], 'fraud': t['fraud'],
             'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'pr_auc': t['fraud'] / t['rows'],
             'false_alerts': 0, 'missed_fraud': t['fraud']})
summary = pd.DataFrame(rows)
summary.to_csv(PROJECT / 'Part_III' / 'Results' / 'fixed_final_evaluation_summary.csv', index=False)
summary.style.format({'precision': '{:.4f}', 'recall': '{:.4f}', 'f1': '{:.4f}', 'pr_auc': '{:.4f}',
                      'rows': '{:,}', 'fraud': '{:,}', 'false_alerts': '{:,}', 'missed_fraud': '{:,}'})

,model,split,rows,fraud,precision,recall,f1,pr_auc,false_alerts,missed_fraud
0,GB + history (final),Training,"5,034,685","23,744",0.9202,0.9621,0.9407,0.9910,"1,981",900
1,GB + history (final),Validation,"1,258,672","2,430",0.9518,0.8770,0.9128,0.9580,108,299
2,GB + history (final),2026 test,"956,022","2,032",0.9533,0.8642,0.9066,0.9610,86,276
3,GB transaction-only,Training,"5,034,685","23,744",0.7098,0.8719,0.7826,0.8989,"8,462","3,042"
4,GB transaction-only,Validation,"1,258,672","2,430",0.4950,0.8000,0.6116,0.7537,"1,983",486
5,GB transaction-only,2026 test,"956,022","2,032",0.4754,0.8332,0.6054,0.7778,"1,868",339
6,Random Forest,Training,"5,034,685","23,744",0.6351,0.8611,0.7310,0.8762,"11,749","3,297"
7,Random Forest,Validation,"1,258,672","2,430",0.4305,0.8000,0.5597,0.6598,"2,572",486
8,Random Forest,2026 test,"956,022","2,032",0.4284,0.8184,0.5624,0.6817,"2,219",369
9,Majority class,2026 test,"956,022","2,032",0.0000,0.0000,0.0000,0.0021,0,"2,032"


## 6. Final model: per-class results and confusion matrix (2026)

In [6]:
r = test_scores['GB + history (final)']
tn, fp, fn, tp = r['tn'], r['fp'], r['fn'], r['tp']
def prf(tp_, fp_, fn_):
    p = tp_ / (tp_ + fp_); rc = tp_ / (tp_ + fn_); return p, rc, 2 * p * rc / (p + rc)
legit, fraud = prf(tn, fn, fp), prf(tp, fp, fn)
per_class = pd.DataFrame({'precision': [legit[0], fraud[0]], 'recall': [legit[1], fraud[1]],
                          'f1': [legit[2], fraud[2]], 'support': [tn + fp, tp + fn]},
                         index=['Legitimate (0)', 'Fraud (1)'])
per_class.loc['Macro average'] = list(per_class.iloc[:2, :3].mean()) + [tn + fp + fn + tp]
display(per_class.round(4))
cm = pd.DataFrame([[tn, fp], [fn, tp]], index=['Actual Legitimate', 'Actual Fraud'],
                  columns=['Predicted Legitimate', 'Predicted Fraud'])
display(cm)
assert cm.values.sum() == r['rows'], 'Confusion matrix total must equal the test size.'
print('Accuracy:', round(r['accuracy'], 4), '| Review workload:', f"{r['review_rate']:.2%}")

,precision,recall,f1,support
Legitimate (0),0.9997,0.9999,0.9998,953990.0
Fraud (1),0.9533,0.8642,0.9066,2032.0
Macro average,0.9765,0.9320,0.9532,956022.0


,Predicted Legitimate,Predicted Fraud
Actual Legitimate,953904,86
Actual Fraud,276,1756


Accuracy: 0.9996 | Review workload: 0.19%


## 7. Subgroup results for the final model (category, gender, age)

In [7]:
test = pd.read_csv(DATA_DIR / 'fraudTest.csv', usecols=['trans_date_trans_time', 'cc_num', 'merchant', 'category',
                   'amt', 'gender', 'dob', 'is_fraud'], dtype={'cc_num': 'string'})
row_scores = pd.read_csv(FINAL / 'test_scores.csv')   # written by evaluate.py, 1-based row numbers
assert len(test) == len(row_scores) and (row_scores.row_number.to_numpy() == np.arange(1, len(test) + 1)).all()
test['score'] = row_scores.model_score.to_numpy()
test['flag'] = row_scores.review_flag.astype(str).str.lower().eq('true').to_numpy()
assert int((test.flag & (test.is_fraud == 1)).sum()) == r['tp'] and int((test.flag & (test.is_fraud == 0)).sum()) == r['fp']
test['ts'] = pd.to_datetime(test.trans_date_trans_time)
age = (test.ts - pd.to_datetime(test.dob)).dt.days / 365.25
test['age_group'] = pd.cut(age, [0, 25, 50, 200], labels=['Under 25', '25-49', '50+'], right=False)

def group_table(column):
    g = test.groupby(column, observed=True)
    out = pd.DataFrame({'transactions': g.size(), 'fraud': g.is_fraud.sum(),
                        'caught': g.apply(lambda x: (x.flag & (x.is_fraud == 1)).sum(), include_groups=False),
                        'false_alerts': g.apply(lambda x: (x.flag & (x.is_fraud == 0)).sum(), include_groups=False)})
    out['recall'] = out.caught / out.fraud
    out['precision'] = out.caught / (out.caught + out.false_alerts)
    out['false_alert_rate'] = out.false_alerts / (out.transactions - out.fraud)
    return out.sort_values('recall')

for column in ['category', 'gender', 'age_group']:
    display(group_table(column).round(4))
fraud_rows = test[test.is_fraud == 1]
print('Median missed fraud $', fraud_rows[~fraud_rows.flag].amt.median(),
      '| median caught fraud $', fraud_rows[fraud_rows.flag].amt.median())
print('Share of fraud under $50 by gender:', fraud_rows.groupby('gender').amt.apply(lambda s: round((s < 50).mean(), 3)).to_dict())

,transactions,fraud,caught,false_alerts,recall,precision,false_alert_rate
category,,,,,,,
travel,30728,33,14,2,0.4242,0.8750,0.0001
health_fitness,66026,30,14,0,0.4667,1.0000,0.0000
food_dining,73538,34,18,0,0.5294,1.0000,0.0000
kids_pets,79589,37,22,2,0.5946,0.9167,0.0000
home,78410,40,24,2,0.6000,0.9231,0.0000
personal_care,66733,46,28,5,0.6087,0.8485,0.0001
misc_pos,62048,61,41,1,0.6721,0.9762,0.0000
grocery_net,37626,49,33,0,0.6735,1.0000,0.0000
entertainment,73623,54,43,2,0.7963,0.9556,0.0000


,transactions,fraud,caught,false_alerts,recall,precision,false_alert_rate
gender,,,,,,,
F,509668,1034,848,53,0.8201,0.9412,0.0001
M,446354,998,908,33,0.9098,0.9649,0.0001


,transactions,fraud,caught,false_alerts,recall,precision,false_alert_rate
age_group,,,,,,,
25-49,560841,1179,1009,37,0.8558,0.9646,0.0001
50+,319762,659,577,48,0.8756,0.9232,0.0002
Under 25,75419,194,170,1,0.8763,0.9942,0.0000


Median missed fraud $ 250.925 | median caught fraud $ 672.855
Share of fraud under $50 by gender: {'F': 0.237, 'M': 0.157}


## 8. Error analysis: misclassified 2026 records

The missed fraud with the median amount and the false alert with the highest score are selected
automatically, then shown with the same card's transactions around them (labels shown for
analysis only, never used as model inputs).

In [8]:
pd.set_option('display.width', 200)
def around(row_number, before='3D', after='2D'):
    x = test.iloc[row_number - 1]
    card = test[test.cc_num == x.cc_num]
    window = card[(card.ts >= x.ts - pd.Timedelta(before)) & (card.ts <= x.ts + pd.Timedelta(after))]
    print(f"Row {row_number}: {x.trans_date_trans_time} {x.category} ${x.amt} | actual fraud={x.is_fraud} | "
          f"score={x.score:.4f} | flagged={x.flag}")
    display(window[['trans_date_trans_time', 'merchant', 'category', 'amt', 'is_fraud', 'score', 'flag']])

missed = test[(test.is_fraud == 1) & ~test.flag]
false_alerts = test[(test.is_fraud == 0) & test.flag]
display(missed.amt.describe().round(2))
if len(missed):
    typical = (missed.amt - missed.amt.median()).abs().idxmin()
    around(typical + 1)        # missed fraud near the median missed amount
if len(false_alerts):
    around(false_alerts.score.idxmax() + 1)   # most confident false alert
print('Missed fraud by category:', missed.category.value_counts().head(6).to_dict())
print('False alerts by category:', false_alerts.category.value_counts().head(6).to_dict())

count     276.00
mean      447.34
std       434.33
min         1.47
25%        18.29
50%       250.92
75%       896.85
max      1393.22
Name: amt, dtype: float64

Row 919687: 2026-12-24 02:06:36 home $250.98 | actual fraud=1 | score=0.9268 | flagged=False


,trans_date_trans_time,merchant,category,amt,is_fraud,score,flag
910857,2026-12-21 13:48:46,"fraud_fraud_Bradtke, Torp and Bahringer",personal_care,125.92,0,0.000072,False
914169,2026-12-22 14:55:08,fraud_fraud_McDermott-Weimann,grocery_pos,184.59,0,0.000002,False
914196,2026-12-22 15:00:27,"fraud_fraud_Nicolas, Hills and McGlynn",entertainment,215.85,0,0.000001,False
916086,2026-12-23 02:44:27,fraud_fraud_Effertz LLC,entertainment,512.10,1,0.753564,False
916136,2026-12-23 03:12:04,fraud_fraud_Shields-Wunsch,health_fitness,21.16,1,0.741823,False
916628,2026-12-23 07:31:49,fraud_fraud_Williamson LLC,misc_pos,7.31,1,0.058567,False
917045,2026-12-23 11:13:57,fraud_fraud_Ullrich Ltd,kids_pets,17.69,1,0.021105,False
918828,2026-12-23 20:47:35,fraud_fraud_Prosacco LLC,personal_care,18.91,1,0.019316,False
919152,2026-12-23 22:21:55,fraud_fraud_Bruen-Yost,misc_pos,10.05,1,0.004330,False
919171,2026-12-23 22:26:30,fraud_fraud_Beier-Hyatt,shopping_pos,819.22,1,0.341203,False


Row 562936: 2026-08-21 00:31:46 shopping_net $1244.81 | actual fraud=0 | score=0.9975 | flagged=True


,trans_date_trans_time,merchant,category,amt,is_fraud,score,flag
562935,2026-08-21 00:31:46,"fraud_fraud_Ruecker, Beer and Collier",shopping_net,1244.81,0,0.997513,True
570421,2026-08-22 20:55:05,fraud_fraud_Champlin-Casper,home,6.41,0,0.000057,False


Missed fraud by category: {'shopping_net': 55, 'shopping_pos': 36, 'misc_net': 30, 'misc_pos': 20, 'travel': 19, 'personal_care': 18}
False alerts by category: {'shopping_net': 36, 'shopping_pos': 21, 'misc_net': 14, 'personal_care': 5, 'home': 2, 'kids_pets': 2}


## 8b. Explaining individual predictions

For one correctly flagged fraud and one false alert, each input is replaced, one at a time, by
the value that is typical for the same card (median, or most common category, over its earlier
legitimate 2026 transactions). The drop in score shows how much that input drove the alert.
Labels are used only to choose the reference transactions, never as model inputs.

In [9]:
from core import REQUIRED, load_bundle, required_columns, validate_input
from history import HISTORY_FEATURES, history_with_context

bundle = load_bundle(FINAL / 'selected_model.joblib')
pipeline, threshold = bundle['pipeline'], bundle['threshold']
pipeline.set_params(features__history='precomputed')
COLUMNS = required_columns(True)

def card_frames(row_number):
    test_all = pd.read_csv(DATA_DIR / 'fraudTest.csv', usecols=COLUMNS + ['is_fraud'], dtype={'cc_num': 'string'})
    card = test_all.iloc[row_number - 1].cc_num
    review = test_all[test_all.cc_num == card]
    parts = [chunk[chunk.cc_num == card] for chunk in pd.read_csv(DATA_DIR / 'fraudTrain.csv', usecols=COLUMNS,
             dtype={'cc_num': 'string'}, chunksize=1_000_000)]
    context = pd.concat(parts)
    features = review[COLUMNS].join(history_with_context(validate_input(context[COLUMNS], history=True),
                                                         validate_input(review[COLUMNS], history=True)))
    return features, review

def explain(row_number, top=6):
    features, review = card_frames(row_number)
    index = row_number - 1
    model_inputs = REQUIRED + HISTORY_FEATURES
    x = features.loc[[index], model_inputs]
    base = pipeline.predict_proba(x)[0, 1]
    # "Typical" value for this card: median (numeric) or most common value (text) over its
    # LEGITIMATE 2026 transactions before this one. Labels are used only to build the reference.
    earlier = features.loc[(features.index < index) & (review.is_fraud == 0), model_inputs]
    rows = []
    for column in model_inputs:
        if column in ('trans_date_trans_time', 'dob'):
            continue
        typical = earlier[column].median() if pd.api.types.is_numeric_dtype(earlier[column]) else earlier[column].mode().iloc[0]
        changed = x.copy(); changed[column] = typical
        rows.append({'feature': column, 'this_transaction': x[column].iloc[0], 'card_typical': typical,
                     'score_if_typical': pipeline.predict_proba(changed)[0, 1]})
    table = pd.DataFrame(rows)
    table['score_drop'] = base - table.score_if_typical
    print(f'Row {row_number}: score {base:.4f} (threshold {threshold:.4f}), actual fraud = {int(review.loc[index, "is_fraud"])}')
    display(table.sort_values('score_drop', ascending=False).head(top).round(4))

explain(922960)   # correctly flagged fraud during the December 23-24 spree
explain(562936)   # false alert: large online purchase after midnight

Row 922960: score 0.9999 (threshold 0.9610), actual fraud = 1


,feature,this_transaction,card_typical,score_if_typical,score_drop
0,amt,345.95,53.765,0.2832,0.7167
6,category,grocery_pos,shopping_net,0.7470,0.2528
20,hist_high_amt_count_72h,5.0,0.0,0.9337,0.0662
25,hist_amt_to_category_mean,1.715474,0.807991,0.9839,0.0160
19,hist_log_amt_sum_24h,7.764759,5.431292,0.9970,0.0028
21,hist_night_count_72h,10.0,2.0,0.9993,0.0006


Row 562936: score 0.9975 (threshold 0.9610), actual fraud = 0


,feature,this_transaction,card_typical,score_if_typical,score_drop
0,amt,1244.81,21.475,0.8141,0.1834
23,hist_hour_share,0.017054,0.065633,0.9784,0.0191
25,hist_amt_to_category_mean,13.114069,0.461261,0.9895,0.0080
12,hist_log_secs_since_prev,12.798479,11.172101,0.9912,0.0063
6,category,shopping_net,shopping_pos,0.9948,0.0027
10,hist_amt_to_prior_mean,19.206146,0.330734,0.9958,0.0017


## 9. Exploratory (post-test): card look-back review

**Exploratory, not part of the model result.** The idea came from errors in the earlier,
defective generation; the 24/72-hour windows were not chosen on validation. It measures a proposed analyst rule: when a
transaction is flagged, also review that card's previous 24 or 72 hours. Before any real use,
the window must be chosen on validation data.

In [10]:
flagged = test[test.flag][['cc_num', 'ts']]
by_card = {cc: g.sort_values('ts') for cc, g in test.groupby('cc_num')}
rows = []
for hours in (24, 72):
    extra = set()
    for cc, f in flagged.groupby('cc_num'):
        card = by_card[cc]
        for ts in f.ts:
            w = card[(card.ts < ts) & (card.ts >= ts - pd.Timedelta(hours=hours)) & (~card.flag)]
            extra.update(w.index)
    e = test.loc[sorted(extra)]
    caught = int((test.flag & (test.is_fraud == 1)).sum()) + int(e.is_fraud.sum())
    alerts = int(test.flag.sum()) + len(e)
    rows.append({'look_back_hours': hours, 'extra_reviews': len(e), 'extra_fraud_found': int(e.is_fraud.sum()),
                 'total_recall': caught / int(test.is_fraud.sum()), 'total_precision': caught / alerts})
pd.DataFrame(rows).round(4)

,look_back_hours,extra_reviews,extra_fraud_found,total_recall,total_precision
0,24,762,258,0.9911,0.7734
1,72,2039,265,0.9946,0.5207


## 10. Summary

- Validation comparison: `results/fixed_validation_comparison.csv`.
- Training, validation and 2026 scores for every model: `results/fixed_final_evaluation_summary.csv`.
- Each model folder holds its `training_score.json` and `final_test.json` evidence.
- All data are synthetic (Sparkov); results do not establish real banking performance.